# JAX search numerical differences

## Goal
Compare batched filtering, Live selection and event/coincidence processing with the original CPU implementations. All inputs are deterministic synthetic arrays; no waveform generation, detector data or complete search is involved. Default JAX calculations stay on the selected device. Independent `reference_operations` opt into original CPU calculations for validation.

Exact checks below require equal shape, dtype and bytes. A small numerical error can still change a threshold decision or tie ordering. Recorded measurements apply to the displayed software and device; rerun on the intended hardware. Native boundaries copy data to the host and cannot be differentiated or traced through JAX.

## Setup
Install PyCBC with JAX, its usual scientific dependencies and a native FFT backend. Choose `device="cpu"` or an available JAX device selector such as `"cuda:0"`. The notebook uses the same native FFT settings for the CPU oracle and validation routes.

In [1]:
import types
import warnings
import platform
from types import SimpleNamespace
import importlib.metadata
import numpy as np
import jax
import jax.numpy as jnp
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", message="Wswiglal-redir-stdio")

from pycbc import scheme
from pycbc.types import FrequencySeries
from pycbc.filter import matched_filter, sigmasq
from pycbc.filter.matchedfilter import LiveBatchMatchedFilter
from pycbc.filter.matchedfilter_jax import batch_matched_filter_bank
from pycbc.waveform.bank import sigma_cached
from pycbc.vetoes.chisq import SingleDetPowerChisq
from pycbc.events import ranking, cuts, veto, coinc
from pycbc.events.eventmgr import EventManager, findchirp_cluster_over_window
from pycbc.events.eventmgr_jax import JAXEventManager
from pycbc.events.stat import QuadratureSumStatistic

device = "cpu"
with scheme.JAXScheme(device) as context:
    hardware = context.jax_device.device_kind
versions = {name: importlib.metadata.version(name)
            for name in ("numpy", "jax", "jaxlib", "lalsuite")}
display({"versions": versions, "device": hardware, "architecture": platform.machine()})


def exact(actual, expected):
    actual, expected = np.asarray(actual), np.asarray(expected)
    assert actual.shape == expected.shape
    assert actual.dtype == expected.dtype
    assert actual.tobytes() == expected.tobytes()


def comparison(actual, expected):
    actual, expected = np.asarray(actual), np.asarray(expected)
    assert actual.shape == expected.shape
    return {"dtype": str(actual.dtype),
            "different_values": int(np.count_nonzero(actual != expected)),
            "max_abs_difference": float(np.max(np.abs(actual - expected), initial=0)),
            "same_bytes": actual.dtype == expected.dtype
                          and actual.tobytes() == expected.tobytes()}


def table(rows):
    keys = list(rows[0])
    text = "| " + " | ".join(keys) + " |\n"
    text += "| " + " | ".join("---" for _ in keys) + " |\n"
    text += "\n".join("| " + " | ".join(str(row[key]) for key in keys) + " |"
                       for row in rows)
    display(Markdown(text))

{'versions': {'numpy': '2.5.2',
  'jax': '0.11.1',
  'jaxlib': '0.11.1',
  'lalsuite': '7.26.15'},
 'device': 'cpu',
 'architecture': 'arm64'}

## Steps

### 1. Batched filtering with fixed grids
The bank has three 65-bin templates on a 0.5 Hz grid. The original CPU oracle filters each template separately. JAX processes the matrix with the same cutoffs, correlation-before-division order and normalization. We compare both complex64 and complex128, with and without PSD weighting.

The original correlation, PSD division, inverse transform and inner-product reduction can each be selected independently. Composing these controls restores the original output and normalization bytes. This does not make a claim about which internal FFT butterfly or machine instruction caused a default difference.

In [2]:
bank_rows = []
for dtype in (np.complex64, np.complex128):
    rng = np.random.default_rng(512)
    templates = (rng.normal(size=(3, 65)) + 1j*rng.normal(size=(3, 65))).astype(dtype)
    strain = (rng.normal(size=65) + 1j*rng.normal(size=65)).astype(dtype)
    real_dtype = np.float32 if dtype == np.complex64 else np.float64
    weights = rng.uniform(0.5, 2.0, 65).astype(real_dtype)
    for weighted in (False, True):
        psd = weights if weighted else None
        options = dict(low_frequency_cutoff=1.3, high_frequency_cutoff=25.9)
        with scheme.CPUScheme():
            data = FrequencySeries(strain, delta_f=0.5)
            spectrum = None if psd is None else FrequencySeries(psd, delta_f=0.5)
            series = [FrequencySeries(row, delta_f=0.5) for row in templates]
            expected = np.stack([matched_filter(row, data, spectrum, **options).numpy()
                                 for row in series])
            expected_norms = np.asarray([sigmasq(row, spectrum, **options)
                                        for row in series])
        reduction = "weighted_inner" if weighted else "inner"
        controls = ("correlate", "divide", "ifft", reduction)
        for label, operations in (("default", ()), ("original components", controls)):
            with scheme.JAXScheme(device, reference_operations=operations) as context:
                result, norms = batch_matched_filter_bank(
                    templates, strain, psd, delta_f=0.5, **options)
                assert result.devices() == norms.devices() == {context.jax_device}
                if operations:
                    exact(result, expected)
                    exact(norms, expected_norms)
                bank_rows.append({"input": np.dtype(dtype).name,
                                  "PSD": weighted, "mode": label,
                                  **comparison(result, expected),
                                  "norm_same_bytes": comparison(norms, expected_norms)["same_bytes"]})
table(bank_rows)

| input | PSD | mode | dtype | different_values | max_abs_difference | same_bytes | norm_same_bytes |
| --- | --- | --- | --- | --- | --- | --- | --- |
| complex64 | False | default | complex64 | 347 | 7.152557373046875e-07 | False | False |
| complex64 | False | original components | complex64 | 0 | 0.0 | True | True |
| complex64 | True | default | complex64 | 364 | 7.251217652992636e-07 | False | False |
| complex64 | True | original components | complex64 | 0 | 0.0 | True | True |
| complex128 | False | default | complex128 | 350 | 1.3368855554576669e-15 | False | False |
| complex128 | False | original components | complex128 | 0 | 0.0 | True | True |
| complex128 | True | default | complex128 | 372 | 1.7763568394002505e-15 | False | False |
| complex128 | True | original components | complex128 | 0 | 0.0 | True | True |

### 2. Complete Live results and the inputs to signal-consistency vetoes
`LiveBatchMatchedFilter.process_data` uses the original duration groups and frequency/PSD grids. A recording adapter delegates power chi-square to the real `SingleDetPowerChisq` calculation and captures its correlation, unscaled peak, normalization and sample index. The sine-Gaussian veto is disabled here, so its output is zero.

The seven filter/selection controls are combined with the two power chi-square controls. Every result column, including template metadata, and every captured veto input is checked against the CPU execution. Power chi-square mechanisms are described in the filtering companion notebook; this example verifies their composition with Live.

In [3]:
class RecordingPowerChisq(SingleDetPowerChisq):
    def __init__(self):
        super().__init__("4", None)
        self.inputs = []

    def values(self, corr, snrv, snr_norm, psd, indices, template):
        self.inputs.append((np.asarray(corr).copy(), np.asarray(snrv).copy(),
                            float(snr_norm), np.asarray(indices), template.id))
        return super().values(corr, snrv, snr_norm, psd, indices, template)


rng = np.random.default_rng(517)
live_templates = (rng.normal(size=(2, 65)) + 1j*rng.normal(size=(2, 65))).astype(np.complex64)
live_templates[:, (0, -1)] = 0
live_strain = (rng.normal(size=65) + 1j*rng.normal(size=65)).astype(np.complex64)
live_strain[[0, -1]] = 0
live_psd = rng.uniform(0.5, 2.0, 65).astype(np.float32)


def run_live(operations=None):
    context = (scheme.CPUScheme() if operations is None
               else scheme.JAXScheme(device, reference_operations=operations))
    with context:
        templates = []
        for index, values in enumerate(live_templates):
            template = FrequencySeries(values, delta_f=0.5)
            template.id = index
            template.approximant = "TaylorF2"
            template.f_lower = template.min_f_lower = 0.5
            template.end_frequency = 32.0
            template.params = np.array((index + 1.,), dtype=[("mass1", np.float64)])
            template.sigmasq = types.MethodType(sigma_cached, template)
            templates.append(template)
        control = LiveBatchMatchedFilter(
            templates, 0.0, "4", SimpleNamespace(values=lambda *args: None),
            maxelements=256)
        recorder = RecordingPowerChisq()
        control.power_chisq = recorder
        data = FrequencySeries(live_strain, delta_f=0.5)
        data.psd = FrequencySeries(live_psd, delta_f=0.5)
        reader = SimpleNamespace(trim_padding=4, blocksize=1, sample_rate=64,
                                 start_time=100.0, overwhitened_data=lambda df: data)
        result = control.process_data(reader)
        return ({name: np.asarray(value).copy() for name, value in result.items()},
                recorder.inputs,
                tuple((tuple(t.id for t in group), tuple(t.delta_f for t in group),
                       tuple(len(t) for t in group)) for group in control.tgroups))


live_controls = ("correlate", "ifft", "squared_norm", "divide", "inner",
                 "abs_arg_max", "live_selection", "power_chisq_bins",
                 "power_chisq_at_points")
native_live, native_veto_inputs, native_geometry = run_live()
default_live, _, default_geometry = run_live(())
controlled_live, controlled_veto_inputs, controlled_geometry = run_live(live_controls)
assert native_geometry == default_geometry == controlled_geometry
assert native_live.keys() == controlled_live.keys()
for name in native_live:
    exact(controlled_live[name], native_live[name])
assert len(native_veto_inputs) == len(controlled_veto_inputs)
for actual, expected in zip(controlled_veto_inputs, native_veto_inputs):
    for got, want in zip(actual, expected):
        exact(got, want)
table([{"column": name, **comparison(default_live[name], native_live[name]),
        "controlled_same_bytes": True} for name in native_live])
print("Exact veto inputs:", len(controlled_veto_inputs), "candidates; geometry:", native_geometry)

| column | dtype | different_values | max_abs_difference | same_bytes | controlled_same_bytes |
| --- | --- | --- | --- | --- | --- |
| mass1 | float64 | 0 | 0.0 | True | True |
| snr | float32 | 1 | 2.384185791015625e-07 | False | True |
| coa_phase | float32 | 2 | 2.384185791015625e-07 | False | True |
| end_time | float64 | 0 | 0.0 | True | True |
| template_id | uint64 | 0 | 0.0 | True | True |
| sigmasq | float32 | 0 | 0.0 | True | True |
| chisq | float32 | 2 | 1.7881393432617188e-06 | False | True |
| chisq_dof | uint32 | 0 | 0.0 | True | True |
| sg_chisq | float32 | 0 | 0.0 | True | True |

Exact veto inputs: 2 candidates; geometry: (((0, 1), (0.5, 0.5), (65, 65)),)


### 3. Hold peak and normalization fixed at a Live threshold
The generic CPU `sigma_cached` reduction returns a Python float. NumPy treats that scalar weakly when it multiplies a float32 magnitude or a complex64 array. JAX's batched normalization is a strong float64 array. It therefore calculates the product in wider precision before storing complex64. These different placements of rounding explain the following boundary case.

The threshold equals the original float32-rounded magnitude product. The original comparison accepts equality; the wider product falls just below that threshold. The same peak and normalization also produce different stored real parts. This is an isolated arithmetic proof; step 2 tests the actual Live selector and its `live_selection` route in the full API.

In [4]:
peak = np.complex64(-0.17656055 + 0.93697107j)
norm = 2.7859016728226313
threshold = float(abs(peak) * norm)
native_scaled = np.array([peak], np.complex64)
native_scaled *= norm
native_magnitude = abs(peak) * norm
with scheme.JAXScheme(device):
    peak_j = jnp.asarray([peak])
    norm_j = jnp.asarray([norm], dtype=jnp.float64)
    wider_scaled = np.asarray((peak_j * norm_j).astype(jnp.complex64))
    wider_magnitude = float((jnp.abs(peak_j) * norm_j)[0])
assert native_magnitude >= threshold
assert wider_magnitude < threshold
assert native_scaled.tobytes() != wider_scaled.tobytes()
table([{"path": "original weak Python scalar", "magnitude": repr(float(native_magnitude)),
        "accept": bool(native_magnitude >= threshold), "stored_complex64": repr(native_scaled[0])},
       {"path": "strong float64 normalization", "magnitude": repr(wider_magnitude),
        "accept": bool(wider_magnitude >= threshold), "stored_complex64": repr(wider_scaled[0])}])
print("Threshold:", repr(threshold))

| path | magnitude | accept | stored_complex64 |
| --- | --- | --- | --- |
| original weak Python scalar | 2.656249523162842 | True | np.complex64(-0.49188036+2.6103094j) |
| strong float64 normalization | 2.656249235074625 | False | np.complex64(-0.49188033+2.6103094j) |

Threshold: 2.656249523162842


### 4. Ranking and event selections
Original `newsnr` and `effsnr` promote their numeric inputs to float64; JAX follows that policy. Fractional powers and division can still depend on the numerical library, compilation and device. This fixed example measures agreement without assuming a mismatch. Their native selectors restore original scalar/array behavior and values.

Event managers apply chi-square and NewSNR cuts, then retain the loudest events within sample intervals. A JAX manager keeps numeric columns on device. Each corresponding original-manager selector isolates the complete selection stage, including integer field casts and tie ordering.

In [5]:
ranking_rows = []
for operation in ("newsnr", "effsnr"):
    snr = np.array([5.500001, 8.25], np.float32)
    chisq = np.array([1.000001, 2.75], np.float32)
    function = getattr(ranking, operation)
    with scheme.CPUScheme():
        expected = function(snr, chisq)
    for label, controls in (("default", ()), ("original operation", (operation,))):
        with scheme.JAXScheme(device, reference_operations=controls):
            actual = function(jnp.asarray(snr), jnp.asarray(chisq))
            if controls:
                exact(actual, expected)
            ranking_rows.append({"operation": operation, "mode": label,
                                 **comparison(actual, expected)})
table(ranking_rows)


def event_manager(cls):
    manager = cls(SimpleNamespace(chisq_bins=True),
                  ["time_index", "snr", "chisq", "chisq_dof"],
                  [int, complex, float, int], array_minsize=8)
    manager.new_template()
    manager.add_template_events(
        ["time_index", "snr", "chisq", "chisq_dof"],
        [np.array([1, 2]), np.array([5+0j, 5+0j]),
         np.array([3., 5.]), np.array([3, 4])])
    manager.finalize_template_events()
    return manager


event_rows = []
for method, selector, options in (
    ("newsnr_threshold", "event_newsnr_threshold", {"threshold": 1}),
    ("chisq_threshold", "event_chisq_threshold", {"value": 2, "num_bins": 4}),
    ("keep_loudest_in_interval", "event_loudest", {"window": 10, "num_keep": 1}),
):
    with scheme.CPUScheme():
        native = event_manager(EventManager)
        # This fixture's complex SNR fields have zero imaginary components.
        # The original loudest selection stores their magnitude in that field.
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", np.exceptions.ComplexWarning)
            getattr(native, method)(**options)
        expected = native.events.copy()
    with scheme.JAXScheme(device, reference_operations=(selector,)) as context:
        candidate = event_manager(JAXEventManager)
        getattr(candidate, method)(**options)
        for name in expected.dtype.names:
            exact(candidate.events[name], expected[name])
            assert candidate.events[name].device == context.jax_device
        event_rows.append({"method": method, "selector": selector,
                           "retained": len(expected), "all_columns_exact": True})
table(event_rows)

| operation | mode | dtype | different_values | max_abs_difference | same_bytes |
| --- | --- | --- | --- | --- | --- |
| newsnr | default | float64 | 0 | 0.0 | True |
| newsnr | original operation | float64 | 0 | 0.0 | True |
| effsnr | default | float64 | 0 | 0.0 | True |
| effsnr | original operation | float64 | 0 | 0.0 | True |

| method | selector | retained | all_columns_exact |
| --- | --- | --- | --- |
| newsnr_threshold | event_newsnr_threshold | 2 | True |
| chisq_threshold | event_chisq_threshold | 2 | True |
| keep_loudest_in_interval | event_loudest | 1 | True |

### 5. Segment ordering and FindChirp integer times
Segment vetoes use half-open intervals. The JAX implementation coalesces unsorted segments and sorts times stably. The original NumPy time sort uses its default quicksort, whose equal-key order can differ. We distinguish equal membership from equal index order.

FindChirp uses greedy clustering and converts times to int32 before comparisons in both implementations. The fractional-time fixture below makes that original conversion observable; the original control also checks its index dtype and bytes.

In [6]:
tied_times = np.tile(np.array([0., 1., 2.]), 12)
with scheme.CPUScheme():
    segment_native = veto.indices_within_times(tied_times, np.array([0.]), np.array([3.]))
with scheme.JAXScheme(device):
    segment_default = veto.indices_within_times(tied_times, np.array([0.]), np.array([3.]))
with scheme.JAXScheme(device, reference_operations=("segment_veto",)):
    segment_controlled = veto.indices_within_times(tied_times, np.array([0.]), np.array([3.]))
    exact(segment_controlled, segment_native)
assert np.array_equal(np.sort(np.asarray(segment_default)), np.sort(segment_native))

times, values = np.array([0.9, 1.95]), np.array([1., 2.])
with scheme.CPUScheme():
    chirp_native = findchirp_cluster_over_window(times, values, 1)
with scheme.JAXScheme(device):
    chirp_default = findchirp_cluster_over_window(times, values, 1)
    exact(chirp_default, chirp_native)
with scheme.JAXScheme(device, reference_operations=("findchirp_cluster",)):
    exact(findchirp_cluster_over_window(times, values, 1), chirp_native)
table([{"operation": "segment veto", "original_indices": segment_native[:12].tolist(),
        "JAX_indices": np.asarray(segment_default)[:12].tolist(), "controlled_exact": True},
       {"operation": "FindChirp", "original_indices": chirp_native.tolist(),
        "JAX_indices": np.asarray(chirp_default).tolist(), "controlled_exact": True}])

| operation | original_indices | JAX_indices | controlled_exact |
| --- | --- | --- | --- |
| segment veto | [0, 27, 21, 18, 15, 24, 30, 9, 12, 6, 33, 3] | [0, 3, 6, 9, 12, 15, 18, 21, 24, 27, 30, 33] | True |
| FindChirp | [1] | [1] | True |

### 6. Coincidence ties and GPS mean rounding
The same ordering issue affects equal-time, equal-statistic coincidences: a stable sort and NumPy's default quicksort can select different tied input indices. Each coincidence stage has an independent native control.

For absolute GPS times, the original two-detector implementation forms the absolute mean in float64, then casts it to NumPy longdouble for slide separation and clustering. JAX subtracts an anchor before forming its mean and uses float64. At a half-ULP boundary, the original mean rounds away an offset that the relative calculation retains. The mean comparison below isolates this mechanism before calling the actual clustering API.

In [7]:
times = np.tile(np.array([10., 11., 12.]), 12)
statistics = np.ones(times.size)
slide_ids = np.zeros(times.size, np.int32)
coincidence_rows = []
for operation, args in (
    ("time_coincidence", (times, np.array([10.]), 0.1)),
    ("cluster_over_time", (statistics, times, 0.1)),
    ("cluster_coincs", (statistics, times, times, slide_ids, 0., 0.1)),
    ("cluster_coincs_multiifo", (statistics, (times, times), slide_ids, 0., 0.1)),
):
    function = getattr(coinc, operation)
    with scheme.CPUScheme():
        expected = function(*args)
    with scheme.JAXScheme(device):
        actual = function(*args)
    with scheme.JAXScheme(device, reference_operations=(operation,)):
        controlled = function(*args)
    pairs = zip(controlled, expected) if operation == "time_coincidence" else [(controlled, expected)]
    for got, want in pairs:
        exact(got, want)
    first_expected = expected[0] if operation == "time_coincidence" else expected
    first_actual = actual[0] if operation == "time_coincidence" else actual
    coincidence_rows.append({"operation": operation,
                             "original_indices": np.asarray(first_expected)[:12].tolist(),
                             "JAX_indices": np.asarray(first_actual)[:12].tolist(),
                             "controlled_exact": True})
table(coincidence_rows)

base = 1e9
first = np.array([base, base + 0.25])
second = np.array([base + np.spacing(base), base + 0.25])
absolute_mean = (first + second) / 2 - base
relative_mean = ((first - base) + (second - base)) / 2
args = (np.array([1., 2.]), first, second, np.zeros(2, np.int32), 0., 0.25)
with scheme.CPUScheme():
    gps_native = coinc.cluster_coincs(*args)
with scheme.JAXScheme(device):
    gps_default = coinc.cluster_coincs(*args)
with scheme.JAXScheme(device, reference_operations=("cluster_coincs",)):
    exact(coinc.cluster_coincs(*args), gps_native)
assert absolute_mean[0] == 0
assert relative_mean[0] == np.spacing(base) / 2
table([{"path": "original absolute mean", "offsets_seconds": absolute_mean.tolist(),
        "selected_indices": gps_native.tolist()},
       {"path": "JAX relative mean", "offsets_seconds": relative_mean.tolist(),
        "selected_indices": np.asarray(gps_default).tolist()}])

| operation | original_indices | JAX_indices | controlled_exact |
| --- | --- | --- | --- |
| time_coincidence | [0, 27, 21, 18, 15, 24, 30, 9, 12, 6, 33, 3] | [0, 3, 6, 9, 12, 15, 18, 21, 24, 27, 30, 33] | True |
| cluster_over_time | [0, 7, 29] | [0, 1, 2] | True |
| cluster_coincs | [0, 7, 29] | [0, 1, 2] | True |
| cluster_coincs_multiifo | [0, 7, 29] | [0, 1, 2] | True |

| path | offsets_seconds | selected_indices |
| --- | --- | --- |
| original absolute mean | [0.0, 0.25] | [0, 1] |
| JAX relative mean | [5.960464477539063e-08, 0.25] | [1] |

### 7. Quadrature sum: isolate expression lowering
The original statistic is `sum(s**2. for s in detector_statistics)**0.5`. NumPy specializes scalar exponents `2.` and `0.5` to multiplication and square root ([NumPy 2.5.2 source](https://github.com/numpy/numpy/blob/v2.5.2/numpy/_core/src/umath/loops_umath_fp.dispatch.c.src#L139-L163)). The current JAX statistic uses multiplication and square root explicitly. In this separate float32 expression experiment, eager JAX's floating exponent `2.` follows a power expression; integer exponent `2` and multiplication follow a different lowering. Compare each against the same NumPy products. JIT compilation can simplify the floating power and eliminate this difference on the displayed device.

We also compare the current public multiplication/square-root statistic and its `quadrature_sum` control. The expression experiment identifies an arithmetic/lowering boundary; it does not identify a particular processor instruction, and its counts can change with JAX or hardware.

In [8]:
first_stat = np.random.default_rng(42).normal(size=100).astype(np.float32)
second_stat = np.random.default_rng(43).normal(size=100).astype(np.float32)
statistic = QuadratureSumStatistic("snr", [], [])
arguments = ([("H1", first_stat), ("L1", second_stat)], None, None, None)
with scheme.CPUScheme():
    quadrature_native = statistic.rank_stat_coinc(*arguments)
with scheme.JAXScheme(device):
    x = jnp.asarray(first_stat)
    expressions = {"eager floating exponent": x ** 2.,
                   "eager integer exponent": x ** 2,
                   "multiplication": x * x,
                   "JIT floating exponent": jax.jit(lambda value: value ** 2.)(x)}
    quadrature_default = statistic.rank_stat_coinc(*arguments)
with scheme.JAXScheme(device, reference_operations=("quadrature_sum",)):
    quadrature_controlled = statistic.rank_stat_coinc(*arguments)
    exact(quadrature_controlled, quadrature_native)
exact(first_stat ** 2., first_stat * first_stat)
floating_ir = str(jax.jit(lambda value: value ** 2.).lower(x).compiler_ir("stablehlo"))
integer_ir = str(jax.jit(lambda value: value ** 2).lower(x).compiler_ir("stablehlo"))
print("StableHLO floating expression contains power:", "stablehlo.power" in floating_ir)
print("StableHLO integer expression contains multiply:", "stablehlo.multiply" in integer_ir)
table([{"expression": label, **comparison(value, first_stat * first_stat)}
       for label, value in expressions.items()])
table([{"statistic": "default quadrature", **comparison(quadrature_default, quadrature_native)},
       {"statistic": "original quadrature", **comparison(quadrature_controlled, quadrature_native)}])

StableHLO floating expression contains power: True
StableHLO integer expression contains multiply: True


| expression | dtype | different_values | max_abs_difference | same_bytes |
| --- | --- | --- | --- | --- |
| eager floating exponent | float32 | 3 | 5.960464477539063e-08 | False |
| eager integer exponent | float32 | 0 | 0.0 | True |
| multiplication | float32 | 0 | 0.0 | True |
| JIT floating exponent | float32 | 0 | 0.0 | True |

| statistic | dtype | different_values | max_abs_difference | same_bytes |
| --- | --- | --- | --- | --- |
| default quadrature | float32 | 0 | 0.0 | True |
| original quadrature | float32 | 0 | 0.0 | True |

## Checks
All native-composition assertions above compare shape, dtype and bytes. The bank proof covers two precisions and both weighting modes. The Live proof covers the complete result schema and inputs to the actual power chi-square calculation. Event/coincidence controls restore the corresponding original implementation while other stages remain JAX.

These fixed component examples establish neither complete-search equivalence nor one universal default error size. Thresholds, tied events, dtype casts and GPS rounding need separate checks even when most output values agree.

## Next steps
Use the same inputs, frequency grids, template metadata, FFT settings and precision for an application comparison. Start with all relevant original controls, verify that result, and remove one control at a time. Conditioning, PSD and transform differences have separate companion notebooks. Template generation is held fixed in these examples.